<h1>RAZ Systems</h1>

## Problem Statement

You're extending the banking/HR assistant into a **401(k) and compensation assistant** that can answer questions, look up your own pay and contribution data, check third-party vendor policy info, and — only when explicitly asked — **draft and send an email summary**, with a human approval step before anything actually goes out.

This is the most complete agent in the curriculum so far. It combines:

1. **Login** — a minimal real authentication step. The agent only ever looks up *your* compensation data, tied to the email you log in with — not an account number you type in.
2. **RAG over HR policy documents** — using **Supabase/pgvector**, the same real vector-database pattern as `hr_rag_app_v2.py` (not the in-memory version from the previous notebook — this is the "do it for real" version).
3. **Compensation lookup** — SQLite, but keyed by your logged-in email instead of an account number.
4. **A mocked third-party 401(k) vendor** (Prudential) — simulated the same way the HR policy documents were originally mocked, since this notebook doesn't have a live Prudential API to call.
5. **An explicit, separate email pipeline** — triggered only when you ask for it (e.g. "email me a summary of my 401k"), built as **three small agents in sequence**: one writes the subject line, one formats the body as HTML, and a final node sends it via SendGrid — with a human approval gate in between formatting and sending.

The graph below has more nodes than anything earlier in this curriculum, but every individual piece is something you've already built. This notebook is about **composition** — wiring previously-separate patterns (RAG, tool-calling, multi-step pipelines, human-in-the-loop) into one coherent agent.

### How to work through this assignment

Every `# TODO` marks something you need to fill in. Everything else — imports, mock data, database schema, class definitions, and node/edge *names* — is given so you can focus on the logic. Run each cell as you complete it before moving to the next; several later cells depend on earlier ones working correctly.


# Below are the skills covered in this project:

**LangGraph state management** — defining a TypedDict state schema with multiple fields beyond just messages (wants_email, subject, html_body, approved)

**Tool-calling with ToolNode** — binding multiple heterogeneous tools to an LLM and routing with tools_condition-style logic

**Real vector-database RAG** — Supabase/pgvector integration: embedding a query, calling a Postgres RPC function, and ranking/formatting results by similarity

**Mocking a third-party API** — simulating an external vendor (Prudential) with keyword matching and a fallback strategy, in the absence of a live API

**Identity-scoped data access** — designing a tool (get_my_compensation) that takes no free-form arguments and always resolves to the logged-in user, preventing prompt-injection-style access to other users' data

**SQLite integration** — CRUD operations keyed by a natural identifier (email) rather than a synthetic account number

**Multi-agent pipelines vs. tool-calling** — recognizing when a task is a fixed-order sequence (subject line → HTML formatting → send) versus a genuine tool-selection decision

**Human-in-the-loop patterns** — a synchronous input()-based approval gate as the simplest form of HITL, without checkpointer/interrupt() machinery

**Three-way conditional routing** — extending the standard two-way (tool vs. end) router into a three-way branch (tool / email pipeline / end), with correct ordering of checks

**Real external service integration** — SendGrid email sending via the actual SDK shape, with graceful fallback/simulation when no API key is present

**Graph composition** — assembling a full StateGraph from previously-independent building blocks (RAG, tool-calling, multi-step pipeline, HITL) into one coherent agent

**(Bonus) Observability** — LangSmith tracing setup, and reflecting on the difference between logging and tracing

### Setup

Packages langgraph langchain-openai langchain-core supabase sendgrid python-dotenv

In [ ]:
import os
import sqlite3
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from supabase import create_client
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

load_dotenv(override=True)

SUPABASE_URL      = os.getenv("SUPABASE_URL","REPLACE_WITH_YOUR_SUPABASE_URL")
SUPABASE_API_KEY  = os.getenv("SUPABASE_API_KEY") 
SENDGRID_API_KEY  = os.getenv("SENDGRID_API_KEY")
FROM_EMAIL        = os.getenv("FROM_EMAIL", "REPLACE_WITH_YOUR_FROM_EMAIL")

llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

supabase = create_client(SUPABASE_URL, SUPABASE_API_KEY) if SUPABASE_URL else None

## 1. Login (given, fully worked)

A minimal, real authentication step — not a graph node, since logging in happens *before* there's anything for a graph to do. A hardcoded user table simulates an identity provider. The point isn't security (this is a teaching notebook), it's establishing **whose email** every subsequent lookup is scoped to, so the compensation tool never needs an account number typed in by the user — it always uses *the logged-in person's own email*.

Nothing to fill in here — this cell is given so the rest of the notebook has something to log into.

In [ ]:
# Simulated identity provider -- in production this would be your SSO/Okta/etc.
USER_DIRECTORY = {
    "numankhan@razsystems.com":  {"password": "numan",  "name": "Numan Khan"},
    "aizakhan@razsystems.com":   {"password": "aiza", "name": "Aiza Khan"},
    "ajazpasha@razsystems.com":  {"password": "ajaz", "name": "Ajaz Pasha"},
}


def login(email: str, password: str) -> str | None:
    """Returns the logged-in user's email if credentials are valid, else None."""
    record = USER_DIRECTORY.get(email.strip().lower())
    if record and record["password"] == password:
        print(f"✅ Logged in as {record['name']} ({email})")
        return email.strip().lower()
    print("❌ Invalid email or password.")
    return None


# Try it
CURRENT_USER_EMAIL = login("ajazpasha@razsystems.com", "ajaz")

## 2. Compensation database — keyed by email instead of account number

Same SQLite pattern as every earlier notebook, but the primary lookup key is now `email`, not an account number, because the whole point of this tool is "what does *the logged-in person* earn and contribute" — never anyone else's data.

The table schema and sample data are given below. **Your job**: implement `save_compensation` so it writes a row into the table.

In [ ]:
if os.path.exists("compensation.db"):
    os.remove("compensation.db")

conn = sqlite3.connect("compensation.db")
c = conn.cursor()
c.execute("""
CREATE TABLE compensation (
    email TEXT PRIMARY KEY,
    name TEXT,
    annual_salary REAL,
    contribution_pct REAL
)
""")
conn.commit()
conn.close()


def save_compensation(email, name, annual_salary, contribution_pct):
    # TODO: open a connection to compensation.db, REPLACE INTO the compensation
    # table with (email, name, annual_salary, contribution_pct), commit, and close.
    pass


save_compensation("numankhan@razsystems.com", "Numan Khan", 118000, 6.0)
save_compensation("aizakhan@razsystems.com", "Aiza Khan", 132500, 8.0)
save_compensation("ajazpasha@razsystems.com", "Ajaz Pasha", 97500, 4.0)

print("✅ compensation.db ready")

## 3. HR policy RAG — Supabase/pgvector, same pattern as `hr_rag_app_v2.py`

This is the real vector-database version, not the in-memory list from the previous notebook. Same RPC call shape, same embedding model, same idea: embed the query, call a Postgres function that does the similarity search, get back ranked chunks.

**This cell assumes you already have**, from the original HR Handbook RAG work:
- A Supabase project with `pgvector` enabled
- An `hr_documents` table populated by the heading-aware chunking notebook
- A `match_hr_documents` RPC function (the same one `hr_rag_app_v2.py` calls)

If you're running this notebook fresh without that infrastructure, `search_hr_policies` below will return "No relevant sections found" rather than erroring — the tool itself doesn't need to change to be testable end-to-end, only its data source needs to exist for real answers.

**Your job**: implement `search_hr_policies` — embed the query, call the RPC, sort the candidates by similarity, and return the top `TOP_K` formatted.

In [ ]:
TABLE_RPC    = "match_hr_documents"
MATCH_THRESH = 0.3
MATCH_COUNT  = 10
TOP_K        = 3


def get_embedding(text: str) -> list:
    return embeddings.embed_query(text)


@tool
def search_hr_policies(query: str) -> str:
    """Search the HR employee handbook (401k policy, benefits, leave, etc.) stored in the vector database for information relevant to the query."""
    if supabase is None:
        return "Supabase is not configured in this environment -- no policy data available."

    # TODO:
    # 1. Get the embedding for `query` using get_embedding().
    # 2. Call supabase.rpc(TABLE_RPC, {...}) with query_embedding, match_threshold=MATCH_THRESH,
    #    match_count=MATCH_COUNT, and .execute() it.
    # 3. Pull candidates out of result.data (default to [] if empty).
    # 4. If there are no candidates, return "No relevant sections found in the handbook."
    # 5. Sort candidates by similarity, descending, and keep the top TOP_K.
    # 6. Format each candidate as "[Page {page_number}, similarity {similarity:.2f}]\n{content}"
    #    and join them with "\n\n---\n\n".
    pass


# Quick check (requires real Supabase config to return real content)
print(search_hr_policies.invoke({"query": "401k matching policy"})[:300])

## 4. The 401(k) vendor lookup — mocked Prudential, same style as the original HR policy mock

No live Prudential API exists for this notebook to call, so this is simulated the same way the HR policy documents were originally mocked: a small hardcoded set of "vendor responses," matched by keyword. In a real system this would be a real API call to your plan administrator; the *shape* of the tool — same `@tool` decorator, same signature, same place in the graph — wouldn't change.

The mock response data is given below. **Your job**: implement `get_401k_vendor_info` — match the topic against the dictionary keys, with a keyword-overlap fallback if nothing matches directly.

In [ ]:
PRUDENTIAL_MOCK_RESPONSES = {
    "matching": (
        "Prudential 401(k) plan: the company matches 100% of the first 3% of salary contributed, "
        "and 50% of the next 2% (up to 4% total match on a 5% contribution). Matching contributions "
        "vest over 3 years on a graded schedule (33% per year)."
    ),
    "vesting": (
        "Vesting schedule: 0% vested in year 1, 33% in year 2, 67% in year 3, 100% vested at the "
        "start of year 4. Your own contributions are always 100% vested immediately."
    ),
    "withdrawal": (
        "Early withdrawals before age 59½ incur a 10% IRS penalty plus ordinary income tax, except "
        "for qualifying hardship withdrawals. Loans against your 401(k) balance are permitted up to "
        "50% of the vested balance or $50,000, whichever is less."
    ),
    "contribution limit": (
        "The 2026 IRS elective deferral limit is $24,000 ($31,500 if age 50+, including catch-up "
        "contributions). Prudential will automatically stop payroll deductions once you reach the limit."
    ),
}


@tool
def get_401k_vendor_info(topic: str) -> str:
    """Get 401(k) plan details from the third-party plan administrator (Prudential) -- matching formula, vesting schedule, withdrawal rules, or contribution limits."""
    # TODO:
    # 1. Lowercase `topic`.
    # 2. Loop over PRUDENTIAL_MOCK_RESPONSES.items() and return the first response (prefixed with
    #    "[Prudential] ") whose key appears in the lowercased topic.
    # 3. If nothing matched, fall back to the response whose key has the most word-overlap with
    #    the topic (hint: max() with a key= based on set intersection of split words), still
    #    prefixed with "[Prudential] ".
    pass


print(get_401k_vendor_info.invoke({"topic": "what is the matching formula?"}))

## 5. The compensation lookup tool — bound to the logged-in user, not a typed-in account number

This is the one tool in this notebook that does **not** take a free-form argument from the LLM the way the others do. It takes no identifying argument at all from the conversation — it always looks up `CURRENT_USER_EMAIL`, the email established at login. This is a deliberate design choice: the LLM should never be in a position to ask for, or be tricked into asking for, someone else's compensation data.

**Your job**: implement `get_my_compensation` — query the `compensation` table for `CURRENT_USER_EMAIL` and format the result.

In [ ]:
@tool
def get_my_compensation() -> str:
    """Look up the logged-in user's own annual salary and 401(k) contribution percentage. Takes no arguments -- always returns the currently logged-in user's data."""
    # TODO:
    # 1. Connect to compensation.db and SELECT name, annual_salary, contribution_pct FROM
    #    compensation WHERE email = CURRENT_USER_EMAIL.
    # 2. If no row is found, return "No compensation record found for the logged-in user."
    # 3. Otherwise compute annual_contribution = salary * (pct / 100) and return a formatted
    #    string with Name, Annual salary, and 401(k) contribution (percent and dollar amount).
    pass


print(get_my_compensation.invoke({}))

## 6. The main agent — three tools, one `ToolNode`

Same pattern as every earlier ToolNode assignment: bind all three tools to the LLM, wrap them in one `ToolNode`, route with `tools_condition`. Nothing new here structurally — this is the part of the graph you've already proven you understand.

The `State` schema and system message are given. **Your job**: implement `agent_node` — invoke `llm_with_tools` with the system message plus the conversation, and return the new message.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    wants_email: bool       # did the user ask to have this emailed?
    subject: str            # filled in by the subject-line agent
    html_body: str          # filled in by the HTML-formatting agent
    approved: bool          # filled in by the human approval gate


tools = [get_my_compensation, search_hr_policies, get_401k_vendor_info]
llm_with_tools = llm.bind_tools(tools)


def agent_node(state: State) -> dict:
    system = SystemMessage(content=(
        "You are a compensation and 401(k) assistant. You have three tools:\n"
        "- get_my_compensation: the logged-in user's own salary and contribution rate (no arguments needed)\n"
        "- search_hr_policies: the company's own HR/401k policy handbook\n"
        "- get_401k_vendor_info: plan details from Prudential (matching, vesting, withdrawals, limits)\n"
        "Use the single most appropriate tool for each question. Be concise and professional. "
        "Do not ask for or guess any account number, employee ID, or another person's email -- "
        "get_my_compensation already knows who is logged in."
    ))
    # TODO: invoke llm_with_tools with [system] + state["messages"] and return
    # {"messages": [response]}
    pass

## 7. Detecting an email request

Before wiring the routing, we need a way to tell "the user wants this answer emailed" apart from "the user just asked a question." This is a small classifier node, run once per turn -- it sets `wants_email` in state, which the router (built next) uses to decide whether to continue the normal tool-calling loop or branch into the email pipeline.

**Your job**: implement `detect_email_request` — a simple keyword check on the latest human message.

In [ ]:
def detect_email_request(state: State) -> dict:
    # TODO:
    # 1. Find the most recent HumanMessage in state["messages"] (there may be none).
    # 2. If there isn't one, return {"wants_email": False}.
    # 3. Lowercase its content and check whether it contains any of these keywords:
    #    "email", "send me", "mail it", "send this".
    # 4. Return {"wants_email": <True/False>}.
    pass

## 8. The email pipeline — three agents in sequence

This is structurally different from the tool-calling loop above: there's no decision about *which* of these three to run, because all three always run, in a fixed order, when this branch is taken. That's the key distinguishing feature versus the `agent`/`tools` loop: tool-calling is "pick zero or more of N options," this pipeline is "always run exactly these 3 steps in this order."

**Agent 1 — subject line.** Given the conversation so far (including whatever the main agent already answered), write a short, professional email subject line.

**Your job**: implement `subject_line_agent`.

In [ ]:
def subject_line_agent(state: State) -> dict:
    print("\n✍️  SUBJECT_LINE_AGENT — drafting a subject line")

    last_answer = state["messages"][-1].content
    prompt = (
        "Write a short, professional email subject line (under 10 words) summarizing this "
        "answer for an employee's compensation/401(k) inquiry. Reply with ONLY the subject line, "
        f"no quotes, no extra text.\n\nAnswer:\n{last_answer}"
    )
    # TODO: call llm.invoke([HumanMessage(content=prompt)]), strip the result, print it,
    # and return {"subject": subject}.
    pass

**Agent 2 — HTML formatting.** Given the same answer and the subject line just produced, format the body as clean HTML suitable for an email client.

**Your job**: implement `html_formatter_agent`.

In [ ]:
def html_formatter_agent(state: State) -> dict:
    print("🎨 HTML_FORMATTER_AGENT — formatting the email body")

    last_answer = state["messages"][-1].content
    prompt = (
        "Format this answer as a clean, professional HTML email body. Use simple inline styling "
        "(no external CSS), a friendly greeting, the information clearly laid out (use <ul>/<li> or "
        "<table> if it helps readability), and a brief sign-off from \"The HR & Benefits Team\". "
        f"Reply with ONLY the HTML, no markdown code fences.\n\nAnswer:\n{last_answer}"
    )
    # TODO: call llm.invoke([HumanMessage(content=prompt)]), strip the result, print its length,
    # and return {"html_body": html_body}.
    pass

## 9. Human-in-the-loop approval gate

A plain console prompt — no LangGraph `interrupt()`/checkpointer machinery, just a node that calls `input()` and records the decision in state. This is the simplest possible human-in-the-loop pattern: the graph pauses naturally because the node itself is waiting on synchronous input, then the router downstream reads `state["approved"]` to decide whether to actually send.

**Your job**: implement `human_approval_node` — print the email for review, ask for a y/n decision, and record it in state.

In [ ]:
def human_approval_node(state: State) -> dict:
    print("\n" + "=" * 60)
    print("📋 EMAIL READY FOR APPROVAL")
    print("=" * 60)
    print(f"Subject: {state['subject']}")
    print(f"To:      {CURRENT_USER_EMAIL}")
    print("-" * 60)
    print(state["html_body"])
    print("=" * 60)

    # TODO:
    # 1. Prompt with input("Send this email? (y/n): "), strip and lowercase the response.
    # 2. Set approved = True if the response is "y", else False.
    # 3. Print "✅ Approved — sending." or "❌ Not approved — email will not be sent." accordingly.
    # 4. Return {"approved": approved}.
    pass

## 10. Sending the email via SendGrid

Real `sendgrid` SDK call shape — `Mail(...)` + `SendGridAPIClient(...).send(...)`, exactly as you'd use it in production. Since this teaching environment likely doesn't have a live `SENDGRID_API_KEY`, the function should check for one and print what *would* be sent instead of erroring, so the graph is fully runnable either way. Drop in a real key and this same code sends a real email — nothing else changes.

**Your job**: implement `send_email_node`.

In [ ]:
def send_email_node(state: State) -> dict:
    # TODO:
    # 1. If state.get("approved") is falsy, print "⏭️  Skipping send — not approved." and return
    #    {"messages": [AIMessage(content="Email was not sent (not approved).")]}.
    # 2. If SENDGRID_API_KEY isn't set, print a message that you're simulating the send (include
    #    CURRENT_USER_EMAIL and state["subject"]), and return
    #    {"messages": [AIMessage(content=f"(Simulated) Email '{state['subject']}' sent to {CURRENT_USER_EMAIL}.")]}.
    # 3. Otherwise, build a Mail(from_email=FROM_EMAIL, to_emails=CURRENT_USER_EMAIL,
    #    subject=state["subject"], html_content=state["html_body"]), send it via
    #    SendGridAPIClient(SENDGRID_API_KEY).send(message), print the response status code, and
    #    return {"messages": [AIMessage(content=f"Email '{state['subject']}' sent to {CURRENT_USER_EMAIL}.")]}.
    pass

## 11. The router — three-way branch

This is the one genuinely new routing concept in this notebook. Earlier graphs only ever needed "call a tool, or finish." This one needs a **third option**: "finished answering, AND the user wants it emailed -- branch into the email pipeline instead of ending."

The order of checks matters: `tools_condition`-style tool-call detection has to be checked first, since the email branch should only ever trigger once the main agent has actually produced a final, tool-free answer -- not partway through gathering information.

**Your job**: implement `route_after_agent`.

In [ ]:
def route_after_agent(state: State) -> str:
    last_message = state["messages"][-1]

    # TODO:
    # 1. If last_message has a non-empty tool_calls attribute (use getattr with a None default),
    #    return "tools" -- a tool call always takes priority.
    # 2. Otherwise, if state.get("wants_email") is truthy, return "email_pipeline".
    # 3. Otherwise, return "__end__".
    pass

## 12. Assemble the full graph

**Your job**: wire up the graph -- add all seven nodes, set the entry point, add the conditional edge out of `agent` using `route_after_agent`, and add the remaining edges (including the loop back from `tools` to `agent`, and the fixed-order chain through the email pipeline).

In [ ]:
graph = StateGraph(State)

# TODO: add_node for each of: detect_email_request, agent, tools (wrap tools in ToolNode(tools)),
# subject_line_agent, html_formatter_agent, human_approval, send_email

# TODO: graph.set_entry_point("detect_email_request")
# TODO: graph.add_edge("detect_email_request", "agent")

# TODO: graph.add_conditional_edges("agent", route_after_agent, {
#     "tools": "tools",
#     "email_pipeline": "subject_line_agent",
#     "__end__": END,
# })
# TODO: graph.add_edge("tools", "agent")  # loop back after any tool call

# TODO: chain the email pipeline in fixed order:
#   subject_line_agent -> html_formatter_agent -> human_approval -> send_email -> END

app = graph.compile()

print("✅ Graph compiled —", list(app.get_graph().nodes.keys()))

## 13. Visualize the graph

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Visualization unavailable in this environment:", e)
    print(app.get_graph().draw_mermaid())

## 14. Run it

**Try a plain question first** — this should stay entirely in the `agent`/`tools` loop and never touch the email pipeline.

In [ ]:
result = app.invoke({
    "messages": [HumanMessage(content="What is my current salary and 401k contribution?")],
    "wants_email": False,
    "subject": "",
    "html_body": "",
    "approved": False,
})
print("\nFinal answer:", result["messages"][-1].content)

In [ ]:
result = app.invoke({
    "messages": [HumanMessage(content="What is my current salary,  401k contribution, 401k early withdrawal and let me know about sick leave?")],
    "wants_email": False,
    "subject": "",
    "html_body": "",
    "approved": False,
})
print("\nFinal answer:", result["messages"][-1].content)

**Now try an explicit email request** — this should produce a subject line, an HTML body, prompt you for approval in the console, and then either send (simulated, unless you've set `SENDGRID_API_KEY`) or skip, depending on what you type.

In [ ]:
result = app.invoke({
    "messages": [HumanMessage(content="Can you email me a summary of my 401k contribution and the Prudential matching policy?")],
    "wants_email": False,
    "subject": "",
    "html_body": "",
    "approved": False,
})
print("\nFinal result:", result["messages"][-1].content)

### Bonus reflection

In 3-5 sentences: pick one thing you saw in the LangSmith trace that **wasn't** visible from this notebook's `print()` statements alone. What does that tell you about the difference between *logging* (what you've been doing with `print()` throughout this curriculum) and *tracing* (what LangSmith adds)?

> _Your answer here._

In [ ]:
# 🌟 BONUS — LangSmith setup
# Requires LANGSMITH_TRACING, LANGSMITH_API_KEY, and (optionally) LANGSMITH_PROJECT
# to already be set in your .env file -- see the markdown cell above.
# Place it on the first cell of your notebook to enable LangSmith tracing for this assignment.
# if needed restart the notebook kernel after adding these to your .env file.

from dotenv import load_dotenv
load_dotenv(override=True)

import os

if os.getenv("LANGSMITH_TRACING") == "true" and os.getenv("LANGSMITH_API_KEY"):
    print("✅ LangSmith tracing is ON")
    print("   Project:", os.getenv("LANGSMITH_PROJECT", "default"))
    print("   Every node in `app` will now appear as a trace at https://smith.langchain.com")
else:
    print("ℹ️  LangSmith tracing is OFF — set LANGSMITH_TRACING=true and LANGSMITH_API_KEY in your .env to enable it.")
    print("    This is a bonus feature; the assignment works correctly either way.")

In [ ]:
result_traced = app.invoke({
    "messages": [HumanMessage(content="Can you email me a summary of my 401k contribution and the Prudential matching policy?")],
    "wants_email": False,
    "subject": "",
    "html_body": "",
    "approved": False,
})
print("\nFinal result:", result_traced["messages"][-1].content)
print("\nIf LangSmith tracing was on, check your project at https://smith.langchain.com for this run's trace.")

## What this notebook covers

| Concept | Where it shows up here | Where you first built it |
|---|---|---|
| Real Supabase/pgvector RAG | `search_hr_policies` | `hr_rag_app_v2.py` |
| Mocked third-party vendor | `get_401k_vendor_info` | the original HR policy mock |
| Identity-scoped lookup (no free-form ID) | `get_my_compensation` takes no arguments at all | new in this notebook |
| One `ToolNode`, multiple heterogeneous tools | `tools = [get_my_compensation, search_hr_policies, get_401k_vendor_info]` | the LangGraph RAG+web+account assignment |
| A fixed-order multi-agent pipeline (not tool-calling) | `subject_line_agent → html_formatter_agent → human_approval → send_email` | new in this notebook |
| Human-in-the-loop | `human_approval_node`'s `input()` gate | new in this notebook |
| A three-way conditional router | `route_after_agent` | extends the two-way pattern from every earlier `decide()`/`router()` |

The most important structural lesson: **not every multi-step process belongs in the tool-calling loop.** `get_my_compensation`, `search_hr_policies`, and `get_401k_vendor_info` are tools because the agent genuinely has to *choose* which one (if any) a question needs. The subject-line → HTML → send sequence is **not** modeled as tools, because there's no choice involved — once the user has asked for an email, all three steps always run, in that exact order.